In [3]:
import sys
print(sys.executable)

/Users/nsemjean/TPBDCC-AGENTIC/.venv/bin/python


In [4]:
print(sys.executable)

/Users/nsemjean/TPBDCC-AGENTIC/.venv/bin/python


In [5]:
import tiktoken
encoding = tiktoken.encoding_for_model("gpt-4o")
print(encoding)


<Encoding 'o200k_base'>


In [6]:
system_message ="""
Perform Sentiment analysis of the review presented in the user message.
The result should be positive or negative. Do not justify your response
"""

In [7]:
tokens = encoding.encode(system_message)
print(len(tokens))
print(tokens)

28
[198, 73751, 31712, 2581, 8450, 328, 290, 3358, 13392, 306, 290, 1825, 3176, 558, 976, 1534, 1757, 413, 8841, 503, 12626, 13, 3756, 625, 14883, 634, 3322, 198]


In [8]:
for token in tokens :
    print(encoding.decode_single_token_bytes(token))

b'\n'
b'Perform'
b' Sent'
b'iment'
b' analysis'
b' of'
b' the'
b' review'
b' presented'
b' in'
b' the'
b' user'
b' message'
b'.\n'
b'The'
b' result'
b' should'
b' be'
b' positive'
b' or'
b' negative'
b'.'
b' Do'
b' not'
b' justify'
b' your'
b' response'
b'\n'


In [9]:
def num_tokens_from_string(string : str, encoding_name : str = 'o200k_base') ->int:
    """ return the number of token in a string"""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [10]:
result = num_tokens_from_string("tiktoken is great !")
print(result)

6


In [12]:
#comment formuler les prompts pour les modèles LLM locaux ollama
# installation : ollama run llama3.2
from langchain_ollama import ChatOllama
from IPython.display import display, Markdown
llm = ChatOllama(model="llama3.2")
response = llm.invoke([{"role" : "system", "content" : "You are a helpfull assistant. the output should be in markdown"}, 
            {"role": "user", "content" : "C'est quoi un agent"}])


In [13]:
display(Markdown(response.content))

# Qu'est-ce qu'un agent ?

Un agent est une entité qui représente et exécute les actions d'une autre entité, souvent pour atteindre un objectif ou réaliser une tâche spécifique.

## Caractéristiques d'un agent

Voici quelques caractéristiques clés qui définissent un agent :

*   **Agence** : Un agent est une entité qui agit de son propre chef et prend des décisions indépendantes.
*   **Objectif** : Un agent a un objectif ou une tâche à réaliser, souvent définit par son créateur ou sa mission.
*   **Réalisation du temps** : Un agent peut être temporellement limité, avec un horizon de réalisation des objectifs spécifique.
*   **Interactions** : Un agent interagit avec son environnement pour atteindre ses objectifs.

LAB 2  : Agents avec Langchain

#LAB 2 - 1 Création d’un nouveau agent sans System Message

In [14]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage
from langchain_ollama import  ChatOllama


#initialiser le modèle
modelOllama = ChatOllama(
    model="llama3.2", 
    temperature=0
)

#initialisation de l'agent qui va travailler avec le,modelèe oollama 
agent = create_agent(model=modelOllama)
question = HumanMessage(content="quelle est la capitale de la lune ? ")
response = agent.invoke(
     {"messages" : question}
     )
print(response['messages'][-1].content)

La Lune n'a pas de capitale. La Lune est un satellite naturel de la Terre et ne possède pas d'administration ou de gouvernement. Elle est simplement un corps céleste qui orbite autour de notre planète.

Il est important de noter que la Lune n'est pas une entité politique ou administrative, mais plutôt un objet géologique et astronomique. Il n'y a pas de ville ou de capitale sur la Lune, car elle n'est pas habitée par des êtres humains ou d'autres espèces.

Si vous avez des questions sur la Lune ou son environnement, je serais ravi de vous aider !


#LAB 2 - 2 Agent avec System Message : un message de contrôle qui définit le
comportement global du modèle

In [64]:
system_prompt =  "Vous êtes un auteur de science-fiction ; créez une capitale à la demande des utilisateurs."
agent_scific = create_agent(
    model= modelOllama, 
    system_prompt=system_prompt
)
response = agent_scific.invoke(
    {"messages" : [question]}
    )

print(response['messages'][-1].content)

La Lune n'a pas de capitale ! En effet, la Lune n'est pas un pays ou un État souverain, mais plutôt un satellite naturel de la Terre.

Cependant, si vous souhaitez créer une capitale fictive sur la Lune, je serais ravi de vous aider à y arriver !

Pour commencer, quels sont les critères que vous souhaitez donner à votre capitale lunaire ? Quelle est son fonction, sa population, ses caractéristiques géographiques ?

Je peux vous aider à créer une ville ou un centre de population sur la Lune qui répond à vos attentes !


LAB 2 - 3. - Agent avec Few-shot learning : une méthode où le modèle
apprend une nouvelle tâche ou classe à partir de quelques
exemples seulement

In [60]:
system_prompt = """
Vous êtes un auteur de science-fiction et vous devez créer une capitale spatiale à la demande d'un utilisateur.
Utilisateur : Quelle est la capitale de Mars ?
Auteur : Marsialis
Utilisateur : Quelle est la capitale de Vénus ?
Auteur : Venusovia
"""

In [61]:
agent_scific = create_agent(
    model=modelOllama, 
    system_prompt=system_prompt
)

agent_scific.invoke({"messages" : [question]})

print(response['messages'][-1].content)

La Lune n'a pas de capitale ! En effet, la Lune n'est pas un pays ou un État souverain, mais plutôt un satellite naturel de la Terre.

Cependant, si vous souhaitez créer une capitale fictive sur la Lune, je serais ravi de vous aider à y arriver !

Pour commencer, quels sont les critères que vous souhaitez donner à votre capitale lunaire ? Quelle est son fonction, sa population, ses caractéristiques géographiques ?

Je peux vous aider à créer une ville ou un centre de population sur la Lune qui répond à vos attentes !


#LAB 2 - 4. Agent avec réponse structurée: une sortie organisée selon un
format prédéfini, plutôt que du texte libre. 

In [71]:
system_prompt = """
Vous êtes un auteur de science-fiction et vous devez créer une capitale spatiale à la demande d'un utilisateur.
Veuillez respecter la structure ci-dessous.
Nom : Nom de la capitale
Localisation : Lieu où elle est située
Ambiance : Description en 2 ou 3 mots
Économie : Principaux secteurs d'activité
"""
agent_scific = create_agent(
    model=modelOllama, 
    system_prompt= system_prompt
)
response = agent_scific.invoke( {"messages" : [question]})
print(response['messages'][-1].content)
print(response['messages'][-1].content_blocks)

Je dois vous corriger ! Il n'y a pas de capitale sur la Lune. La Lune est un satellite naturel de la Terre et n'a pas de gouvernement ou d'organisation humaine permanente.

Cependant, si vous souhaitez créer une capitale spatiale fictive sur la Lune, je serais ravi de vous aider ! Voici une proposition :

**Nom :** Lunaria
**Localisation :** Surface de la Lune, dans le cratère de Clavius
**Ambiance :** Futuriste et Spacieuse

En tant que capitale spatiale, Lunaria serait un centre de commerce, de recherche et d'exploration pour les missions humaines sur la Lune. Elle serait équipée de bases scientifiques, de laboratoires de recherche et de infrastructures pour soutenir les activités spatiales.

**Économie :**

* Secteur principal : Exploration spatiale
* Secteur secondaire : Commerce interplanétaire
* Secteur tertiaire : Recherche scientifique

Lunaria serait un lieu de rencontre pour les nations spatiales et les entreprises qui cherchent à explorer et à développer l'espace. Elle serai

#LAB 2 - 5. Agent avec réponse structurée en utilisant BaseModel : rendre la
réponse facile à exploiter automatiquement par un programme ou
un système.

In [ ]:
from pydantic import BaseModel
class CapitalInfo(BaseModel) : 
    nom : str
    Localisation : str
    Ambiance : str
    Economie : str

system_prompt = """
Vous êtes un auteur de science-fiction et vous devez créer une capitale spatiale à la demande d'un utilisateur.
Veuillez respecter la structure ci-dessous.
Nom : Nom de la capitale
Localisation : Lieu où elle est située
Ambiance : Description en 2 ou 3 mots
Économie : Principaux secteurs d'activité
"""
agent_scific = create_agent(
    model=modelOllama, 
    system_prompt=system_prompt, 
    response_format=CapitalInfo
 )

question = HumanMessage("quelle est la capitale de la lune ?")

response = agent_scific.invoke(
    {"messages" : [question]}
)

print(response['structured_response'])

nom='Lune' Localisation='Lune' Ambiance='Hostile' Economie='Exploration'


LAB 2-6. Les tools (outils) : une fonction externe que le modèle peut
appeler pour effectuer une action précise qu’il ne peut pas faire
seul

In [75]:
from langchain.tools import tool

@tool("meteo_capital")
def meteo_capital(ville : str ) -> str : 
    """
        Donne la météo d'une capitale (valeurs fixes pour test).
        Args:
        ville: nom de la capitale
    """
    print("tool météo capital utilisé")

    #logique métier - récupération des donnéees
    temperature = 45
    humidite = 10
    pression = 68

    return (
        f"Météo à {ville} :"
        f"Température {temperature}°C"
        f"humidité  {humidite}%"
        f"pression {pression}hPa"
    )

In [76]:
#Ajout du Tool à l’agent
system_prompt = "Utilise les tools pour répondre aux questions"

#initialisation de l'agent avec le tool
agent_scific = create_agent(
    model=modelOllama, 
    system_prompt = system_prompt, 
    tools= [meteo_capital]
)
question = HumanMessage(content="quelle est la météo de la capitale lunaire ? ")

response = agent_scific.invoke(
    {"messages":[question]}
)
print(response['messages'][-1].content)

tool météo capital utilisé
La météo à lunaire est la suivante :

**Température :** 45°C
**Humidité :** 10%
**Pression atmosphérique :** 68 hPa

Il fait très chaud et sec sur la Lune !


#LAB 2.7- Agent sans Web Search Tool

In [77]:
agent = create_agent(
    model=modelOllama
)
question = HumanMessage(content="vos connaissances en matières d'apprentissage sont-elles à jour ?")

response = agent.invoke(
    {"messages":[question]}
)
print(response['messages'][-1].content)

Mes connaissances en matière d'apprentissage sont basées sur les dernières recherches et avancées jusqu'en décembre 2023. Cependant, la recherche dans ce domaine est constamment évolutive et nouvelle, et de nouvelles découvertes peuvent être faites à tout moment.

Je peux vous fournir des informations sur les théories et les méthodes d'apprentissage courantes, telles que l'apprentissage supervisé, l'apprentissage non supervisé, l'apprentissage par renforcement, etc. Je peux également discuter de techniques spécifiques telles que le deep learning, la réseaux de neurones convolutives (CNN), les réseaux de neurones recurrentes (RNN) et les modèles d'apprentissage automatique.

Cependant, si vous cherchez des informations sur les dernières avancées ou les recherches en cours, je peux vous dire que mes connaissances peuvent ne pas être à jour. Il est possible que de nouvelles découvertes aient été faites depuis décembre 2023 et que ces informations ne soient pas encore intégrées dans mon sy

#LAB 2.8 - Agent avec Web Search Tool

In [89]:
from typing import Dict, Any
from tavily import TavilyClient
from dotenv import load_dotenv
import os

os.environ["TAVILY_API_KEY"] = "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"
load_dotenv()
tavily_client  = TavilyClient()

@tool("web_search")
def web_search(query: str) -> Dict[str, Any]: 
    """
        
    """
    return tavily_client.search(query)

web_search.invoke("qui est le président actuel du Cameroun ? ")

#sortie de l'agent
agent = create_agent(
    model=modelOllama, 
    tools=[web_search]
)

question  = HumanMessage(content="qui est le président actuel du Cameroun ?")

response = agent.invoke(
    {"messages":[question]}
)

print(response['messages'][-1].content)



Désolé pour la erreur ! Le président actuel du Cameroun est Paul Biya, il a été élu en 1982 et est toujours en fonction depuis. Il est le plus longue durée de règne d'un président en Afrique.


#LAB 2.9 - Agent sans mémoire

In [90]:
agent = create_agent(
    model=modelOllama
)
question = HumanMessage(content="Bonjour, mon nom est Sami et je suis un développeur.")

response = agent.invoke(
    {"messages":[question]}
)
question = HumanMessage(content="Quel est mon métier ?")
response = agent.invoke(
    {"messages":[question]}
)
print(response['messages'][-1].content)

Je serais ravi de t'aider à découvrir ton métier !

Pour commencer, peux-tu me donner quelques informations sur toi-même ?

* Quels sont tes intérêts ?
* Quelles sont tes compétences ?
* As-tu déjà fait des études ou des formations ?
* Qu'est-ce que tu aimes faire dans ta vie quotidienne ?

En répondant à ces questions, nous pourrons commencer à explorer ensemble les différentes options de métiers qui pourraient correspondre à tes talents et à tes passions !

Et si tu veux, on peut également jouer un jeu pour t'aider à découvrir ton métier. Je peux te poser des questions sur toi-même, comme :

* Qu'est-ce que tu aimes faire pendant les vacances ?
* Quels sont tes objectifs à court et à long terme ?
* Qu'est-ce que tu penses être ton "métier idéal" ?

N'hésite pas à partager tes réponses, je suis là pour t'aider !


#LAB2.10 - Agent avec mémoire

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=modelOllama, 
    #memmoure
    checkpointer=InMemorySaver()
)
#clé de configuration pour la mémoire
config = {"configurable": {"thread_id": "1"}}

question = HumanMessage(content="Bonjour, mon nom est Sami et je suis un développeur.")
response = agent.invoke(
    {"messages": [question]},
    config 
    
)
question = HumanMessage(content="Quel est mon métier ?")
response = agent.invoke(
    {"messages":[question]}, 
    config
)
print(response['messages'][-1].content)




Vous êtes un développeur ! C'est un métier très varié et passionnant, car vous avez l'occasion de créer des solutions innovantes et de résoudre des problèmes complexes pour les entreprises et les individus. En tant que développeur, vous travaillez souvent avec des langages de programmation, des frameworks et des bases de données pour concevoir et développer des applications, des sites web ou des systèmes informatiques.

Qu'est-ce que vous développez actuellement ? Un site web, une application mobile, un système d'information... ?
